# SAM3 TTD Few-Shot Experiment Runner

Run the **Experiment Selection** cell first. The notebook supports two workflows:

1. Single experiment: set `EXPERIMENT_KEY`, then call `run_train(EXPERIMENT_KEY)` / `run_eval(EXPERIMENT_KEY)` from the helper cell.
2. Batch experiment: run the **Batch Train + Eval** cell to process all stable 5 / 10 / 25 / 50-shot experiments for `Single-TB`, `Shift-TA_TC-to-TB_10pct`, and `Shift-TA_TB-to-TC_10pct`.

The batch order intentionally runs all `Single-TB` YAMLs first. This matches the safer ordering used after the earlier dtype / gradient issues.


In [1]:
# Experiment Selection
from pathlib import Path

PROJECT_ROOT = Path("/users/7/yu001011/csci5527")
SAM3_WORK_ROOT = PROJECT_ROOT / "CSCI5527-final" / "SAM3"
SAM3_REPO_ROOT = PROJECT_ROOT / "sam3"

# Experiment key constants. These let you write EXPERIMENT_KEY without quotes.
single_tb_5shot_stable_lowlr_v1 = "single_tb_5shot_stable_lowlr_v1"
single_tb_10shot_stable_lowlr_v1 = "single_tb_10shot_stable_lowlr_v1"
single_tb_25shot_stable_lowlr_v1 = "single_tb_25shot_stable_lowlr_v1"
single_tb_50shot_stable_lowlr_v1 = "single_tb_50shot_stable_lowlr_v1"
shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1 = "shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1"
shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1 = "shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1"
shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1 = "shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1"
shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1 = "shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1"
shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1 = "shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1"
shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1 = "shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1"
shift_ta_tb_to_tc_10pct_25shot_stable_lowlr_v1 = "shift_ta_tb_to_tc_10pct_25shot_stable_lowlr_v1"
shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1 = "shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1"
single_tb_5shot_clean_fp32_v1 = "single_tb_5shot_clean_fp32_v1"
single_tb_10shot_clean_fp32_v1 = "single_tb_10shot_clean_fp32_v1"
single_tb_25shot_clean_fp32_v1 = "single_tb_25shot_clean_fp32_v1"
single_tb_50shot_clean_fp32_v1 = "single_tb_50shot_clean_fp32_v1"
shift_ta_tc_to_tb_10pct_5shot_clean_fp32_v1 = "shift_ta_tc_to_tb_10pct_5shot_clean_fp32_v1"
shift_ta_tc_to_tb_10pct_10shot_clean_fp32_v1 = "shift_ta_tc_to_tb_10pct_10shot_clean_fp32_v1"
shift_ta_tc_to_tb_10pct_25shot_clean_fp32_v1 = "shift_ta_tc_to_tb_10pct_25shot_clean_fp32_v1"
shift_ta_tc_to_tb_10pct_50shot_clean_fp32_v1 = "shift_ta_tc_to_tb_10pct_50shot_clean_fp32_v1"
shift_ta_tb_to_tc_10pct_5shot_clean_fp32_v1 = "shift_ta_tb_to_tc_10pct_5shot_clean_fp32_v1"
shift_ta_tb_to_tc_10pct_10shot_clean_fp32_v1 = "shift_ta_tb_to_tc_10pct_10shot_clean_fp32_v1"
shift_ta_tb_to_tc_10pct_25shot_clean_fp32_v1 = "shift_ta_tb_to_tc_10pct_25shot_clean_fp32_v1"
shift_ta_tb_to_tc_10pct_50shot_clean_fp32_v1 = "shift_ta_tb_to_tc_10pct_50shot_clean_fp32_v1"

# Single-experiment default. Change this when you want to inspect one run with the IoU cell.
EXPERIMENT_KEY = single_tb_5shot_stable_lowlr_v1

EXPERIMENT_GROUPS = {
    "single_tb": "Single-TB",
    "shift_ta_tc_to_tb_10pct": "Shift-TA_TC-to-TB_10pct",
    "shift_ta_tb_to_tc_10pct": "Shift-TA_TB-to-TC_10pct",
}

# Important: keep Single-TB first. Running these YAMLs first avoids the earlier dtype/gradient issue.
BATCH_GROUP_ORDER = [
    "single_tb",
    "shift_ta_tc_to_tb_10pct",
    "shift_ta_tb_to_tc_10pct",
]
BATCH_SHOTS = [5, 10, 25, 50]
BATCH_RUN_TAG = "stable_lowlr_v1"

EXPERIMENTS = {}

def register_experiment(group_key, shot_per_class, run_tag):
    experiment_name = EXPERIMENT_GROUPS[group_key]
    config_stem = f"ttd_{group_key}_{shot_per_class}shot_textseg_{run_tag}"
    run_root = SAM3_WORK_ROOT / "fewshot_data" / experiment_name / f"{shot_per_class}_shot_per_class"
    run_dir = SAM3_WORK_ROOT / "outputs" / "fewshot_runs" / experiment_name / f"{shot_per_class}_shot_per_class_{run_tag}"
    key = f"{group_key}_{shot_per_class}shot_{run_tag}"
    EXPERIMENTS[key] = {
        "key": key,
        "experiment_name": experiment_name,
        "group_key": group_key,
        "shot_per_class": shot_per_class,
        "run_tag": run_tag,
        "run_root": run_root,
        "run_dir": run_dir,
        "train_config": f"configs/ttd_fewshot/{config_stem}.yaml",
        "test_config": f"configs/ttd_fewshot/{config_stem}_test_eval.yaml",
    }

for group_key in EXPERIMENT_GROUPS:
    for shot_per_class in (5, 10, 25, 50):
        register_experiment(group_key, shot_per_class, "stable_lowlr_v1")
        register_experiment(group_key, shot_per_class, "clean_fp32_v1")

BATCH_EXPERIMENT_KEYS = [
    f"{group_key}_{shot_per_class}shot_{BATCH_RUN_TAG}"
    for group_key in BATCH_GROUP_ORDER
    for shot_per_class in BATCH_SHOTS
]

def set_experiment(experiment_key):
    if experiment_key not in EXPERIMENTS:
        available = "\n".join(f"  - {key}" for key in sorted(EXPERIMENTS))
        raise KeyError(f"Unknown EXPERIMENT_KEY: {experiment_key}\nAvailable keys:\n{available}")

    experiment = EXPERIMENTS[experiment_key]
    train_config = experiment["train_config"]
    test_config = experiment["test_config"]
    run_root = experiment["run_root"]
    run_dir = experiment["run_dir"]
    checkpoint = run_dir / "checkpoints" / "checkpoint.pt"
    gt_json = run_root / "test" / "_annotations.coco.json"
    pred_json = run_dir / "dumps" / "ttd" / "test" / "coco_predictions_segm.json"

    train_config_path = SAM3_REPO_ROOT / "sam3" / "train" / train_config
    test_config_path = SAM3_REPO_ROOT / "sam3" / "train" / test_config
    for label, path in {
        "train config": train_config_path,
        "test config": test_config_path,
        "dataset root": run_root,
        "test ground truth": gt_json,
    }.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing {label}: {path}")

    globals().update({
        "EXPERIMENT_KEY": experiment_key,
        "EXPERIMENT": experiment,
        "TRAIN_CONFIG": train_config,
        "TEST_CONFIG": test_config,
        "RUN_ROOT": run_root,
        "RUN_DIR": run_dir,
        "CHECKPOINT": checkpoint,
        "GT_JSON": gt_json,
        "PRED_JSON": pred_json,
        "TRAIN_CONFIG_PATH": train_config_path,
        "TEST_CONFIG_PATH": test_config_path,
    })
    return experiment

EXPERIMENT = set_experiment(EXPERIMENT_KEY)

print("Selected experiment:", EXPERIMENT_KEY)
print("Experiment name:", EXPERIMENT["experiment_name"])
print("Shot per class:", EXPERIMENT["shot_per_class"])
print("Run tag:", EXPERIMENT["run_tag"])
print("Training config:", TRAIN_CONFIG)
print("Test eval config:", TEST_CONFIG)
print("Dataset root:", RUN_ROOT)
print("Output dir:", RUN_DIR)
print("\nBatch run order:")
for idx, key in enumerate(BATCH_EXPERIMENT_KEYS, start=1):
    exp = EXPERIMENTS[key]
    print(f"{idx:02d}. {key} -> {exp['experiment_name']} / {exp['shot_per_class']}-shot")


Selected experiment: single_tb_5shot_stable_lowlr_v1
Experiment name: Single-TB
Shot per class: 5
Run tag: stable_lowlr_v1
Training config: configs/ttd_fewshot/ttd_single_tb_5shot_textseg_stable_lowlr_v1.yaml
Test eval config: configs/ttd_fewshot/ttd_single_tb_5shot_textseg_stable_lowlr_v1_test_eval.yaml
Dataset root: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/fewshot_data/Single-TB/5_shot_per_class
Output dir: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/5_shot_per_class_stable_lowlr_v1

Batch run order:
01. single_tb_5shot_stable_lowlr_v1 -> Single-TB / 5-shot
02. single_tb_10shot_stable_lowlr_v1 -> Single-TB / 10-shot
03. single_tb_25shot_stable_lowlr_v1 -> Single-TB / 25-shot
04. single_tb_50shot_stable_lowlr_v1 -> Single-TB / 50-shot
05. shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1 -> Shift-TA_TC-to-TB_10pct / 5-shot
06. shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1 -> Shift-TA_TC-to-TB_10pct / 10-shot
07. shift_ta_tc_to_tb_10pct_25shot_stabl

In [2]:
# Train/Eval Helpers
import os
import subprocess
import sys
from datetime import datetime

if "EXPERIMENTS" not in globals():
    raise RuntimeError("Run the Experiment Selection cell first.")

RUN_ENV = os.environ.copy()
RUN_ENV.setdefault("HYDRA_FULL_ERROR", "1")

def _train_cmd(config):
    return [
        sys.executable,
        "-m", "sam3.train.train",
        "-c", config,
        "--use-cluster", "0",
        "--num-gpus", "1",
    ]

def _run_sam3_command(cmd, label, experiment_key):
    print("=" * 100)
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {label}: {experiment_key}")
    print("Command:", " ".join(cmd))
    return subprocess.run(cmd, check=True, cwd=str(SAM3_REPO_ROOT), env=RUN_ENV)

def run_train(experiment_key=None, skip_existing=True):
    experiment_key = experiment_key or EXPERIMENT_KEY
    experiment = set_experiment(experiment_key)
    print("Selected experiment:", experiment_key)
    print("Training config:", TRAIN_CONFIG)
    print("Output dir:", RUN_DIR)

    if skip_existing and CHECKPOINT.exists():
        print("Skipping train because checkpoint already exists:", CHECKPOINT)
        return "skipped"

    _run_sam3_command(_train_cmd(TRAIN_CONFIG), "TRAIN", experiment_key)
    print("Checkpoint:", CHECKPOINT)
    print("Checkpoint exists:", CHECKPOINT.exists())
    return "trained"

def run_eval(experiment_key=None, skip_existing=True):
    experiment_key = experiment_key or EXPERIMENT_KEY
    experiment = set_experiment(experiment_key)
    print("Selected experiment:", experiment_key)
    print("Test eval config:", TEST_CONFIG)
    print("Using checkpoint:", CHECKPOINT)

    if not CHECKPOINT.exists():
        raise FileNotFoundError(f"Train checkpoint not found yet: {CHECKPOINT}")
    if skip_existing and PRED_JSON.exists() and PRED_JSON.stat().st_size > 2:
        print("Skipping eval because prediction file already exists:", PRED_JSON)
        return "skipped"

    _run_sam3_command(_train_cmd(TEST_CONFIG), "TEST EVAL", experiment_key)
    print("Prediction file:", PRED_JSON)
    print("Prediction exists:", PRED_JSON.exists())
    return "evaluated"

def run_train_and_eval_many(
    experiment_keys,
    do_train=True,
    do_eval=True,
    skip_existing_train=True,
    skip_existing_eval=True,
    stop_on_error=True,
):
    results = []
    for idx, experiment_key in enumerate(experiment_keys, start=1):
        experiment = EXPERIMENTS[experiment_key]
        print("\n" + "#" * 100)
        print(
            f"Batch item {idx}/{len(experiment_keys)}: {experiment_key} "
            f"({experiment['experiment_name']}, {experiment['shot_per_class']}-shot)"
        )
        train_status = "not_requested"
        eval_status = "not_requested"
        error = None
        try:
            if do_train:
                train_status = run_train(experiment_key, skip_existing=skip_existing_train)
            if do_eval:
                eval_status = run_eval(experiment_key, skip_existing=skip_existing_eval)
        except Exception as exc:
            error = repr(exc)
            print(f"ERROR in {experiment_key}: {error}")
            if stop_on_error:
                raise
        results.append({
            "experiment_key": experiment_key,
            "experiment_name": experiment["experiment_name"],
            "shot_per_class": experiment["shot_per_class"],
            "run_tag": experiment["run_tag"],
            "train_status": train_status,
            "eval_status": eval_status,
            "checkpoint": str(CHECKPOINT),
            "prediction_json": str(PRED_JSON),
            "error": error,
        })
    return results


In [3]:
# Batch Train + Eval: all stable few-shot experiments
# This runs Single-TB first, then the two domain-shift combinations.
# Set SKIP_EXISTING_* to False if you want to force reruns.

if "BATCH_EXPERIMENT_KEYS" not in globals():
    raise RuntimeError("Run the Experiment Selection cell first.")
if "run_train_and_eval_many" not in globals():
    raise RuntimeError("Run the Train/Eval Helpers cell first.")

BATCH_DO_TRAIN = True
BATCH_DO_EVAL = True
SKIP_EXISTING_TRAIN = True
SKIP_EXISTING_EVAL = True
STOP_ON_ERROR = True

batch_results = run_train_and_eval_many(
    BATCH_EXPERIMENT_KEYS,
    do_train=BATCH_DO_TRAIN,
    do_eval=BATCH_DO_EVAL,
    skip_existing_train=SKIP_EXISTING_TRAIN,
    skip_existing_eval=SKIP_EXISTING_EVAL,
    stop_on_error=STOP_ON_ERROR,
)

try:
    import pandas as pd
    display(pd.DataFrame(batch_results))
except Exception:
    for row in batch_results:
        print(row)



####################################################################################################
Batch item 1/12: single_tb_5shot_stable_lowlr_v1 (Single-TB, 5-shot)
Selected experiment: single_tb_5shot_stable_lowlr_v1
Training config: configs/ttd_fewshot/ttd_single_tb_5shot_textseg_stable_lowlr_v1.yaml
Output dir: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/5_shot_per_class_stable_lowlr_v1
Skipping train because checkpoint already exists: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/5_shot_per_class_stable_lowlr_v1/checkpoints/checkpoint.pt
Selected experiment: single_tb_5shot_stable_lowlr_v1
Test eval config: configs/ttd_fewshot/ttd_single_tb_5shot_textseg_stable_lowlr_v1_test_eval.yaml
Using checkpoint: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/5_shot_per_class_stable_lowlr_v1/checkpoints/checkpoint.pt
Skipping eval because prediction file already exists: /users/7/yu001011/csc

/users/7/yu001011/csci5527/sam3/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/users/7/yu001011/csci5527/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


###################### Train App Config ####################
paths:
  dataset_root: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/fewshot_data/Single-TB/10_shot_per_class
  experiment_log_dir: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/10_shot_per_class_stable_lowlr_v1
  bpe_path: /users/7/yu001011/csci5527/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz
  sam3_checkpoint: /users/7/yu001011/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt
ttd_fewshot:
  experiment_name: Single-TB
  shot_per_class: 10
  prompt_text: crack
  train_img_folder: ${paths.dataset_root}/train/images
  train_ann_file: ${paths.dataset_root}/train/_annotations.coco.json
  val_img_folder: ${paths.dataset_root}/val/images
  val_ann_file: ${paths.dataset_root}/val/_annotations.coco.json
  test_img_folder: ${paths.dataset_root}/test/images
  test_ann_file: ${paths.dataset_root}/test/_annotations.coco.json
scratch:
  enable_segme


############################################################
Experiment Log Dir:
/users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Single-TB/10_shot_per_class_stable_lowlr_v1
INFO 2026-05-03 11:18:52,774 train_utils.py: 106: MACHINE SEED: 492
INFO 2026-05-03 11:18:52,862 train_utils.py: 152: Logging ENV_VARIABLES
INFO 2026-05-03 11:18:52,862 train_utils.py: 153: BASH_FUNC__module_raw%%=() {  unset _mlre _mlIFS;
 if [ -n "${IFS+x}" ]; then
 _mlIFS=$IFS;
 fi;
 IFS=' ';
 for _mlv in ${MODULES_RUN_QUARANTINE:-};
 do
 if [ "${_mlv}" = "${_mlv##*[!A-Za-z0-9_]}" ] && [ "${_mlv}" = "${_mlv#[0-9]}" ]; then
 if [ -n "$(eval 'echo ${'"$_mlv"'+x}')" ]; then
 _mlre="${_mlre:-}__MODULES_QUAR_${_mlv}='$(eval 'echo ${'"$_mlv"'}')' ";
 fi;
 _mlrv="MODULES_RUNENV_${_mlv}";
 _mlre="${_mlre:-}${_mlv}='$(eval 'echo ${'"$_mlrv"':-}')' ";
 fi;
 done;
 if [ -n "${_mlre:-}" ]; then
 _mlre="${_mlre:-}__MODULES_QUARANTINE_SET=1 ";
 eval "$(eval ${_mlre} /usr/bin/tclsh '/usr/share/Modules/libex

INFO 2026-05-03 11:19:11,643 trainer.py:1150: ====================
INFO 2026-05-03 11:19:11,709 trainer.py:1151: Summary for model <class 'sam3.model.sam3_image.Sam3Image'>
INFO 2026-05-03 11:19:11,713 trainer.py:1152: Model is Sam3Image(
  (backbone): SAM3VLBackbone(
    (vision_backbone): Sam3DualViTDetNeck(
      (trunk): ViT(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        )
        (blocks): ModuleList(
          (0): Block(
            (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): Attention(
              (qkv): Linear(in_features=1024, out_features=3072, bias=True)
              (proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (ls1): Identity()
            (drop_path): Identity()
            (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=1024, 

...

/users/7/yu001011/csci5527/.venv/lib/python3.11/site-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W503 13:53:27.049634644 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


Raw dataset length = 38
INFO 2026-05-03 13:53:27,332 trainer.py: 437: Resuming training from /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Shift-TA_TB-to-TC_10pct/50_shot_per_class_stable_lowlr_v1/checkpoints/checkpoint.pt
INFO 2026-05-03 13:53:54,751 train_utils.py: 269: Val Epoch: [4][ 0/38] | Batch Time: 20.39 (20.39) | Data Time: 0.08 (0.08) | Mem (GB): 42.00 (42.00/42.00) | Time Elapsed: 00d 00h 18m | Losses/val_all_loss: 0.00e+00 (0.00e+00) | Losses/val_default_loss: 0.00e+00 (0.00e+00) | val_ttd/segmentation/: 0.0000
INFO 2026-05-03 13:54:02,236 train_utils.py: 269: Val Epoch: [4][10/38] | Batch Time: 0.71 (2.53) | Data Time: 0.03 (0.05) | Mem (GB): 7.00 (10.18/42.00) | Time Elapsed: 00d 00h 19m | Losses/val_all_loss: 0.00e+00 (0.00e+00) | Losses/val_default_loss: 0.00e+00 (0.00e+00) | val_ttd/segmentation/: 0.0000
INFO 2026-05-03 13:54:09,669 train_utils.py: 269: Val Epoch: [4][20/38] | Batch Time: 0.70 (1.68) | Data Time: 0.03 (0.06) | Mem (GB): 7.00 (8.6

[rank0]:[W503 13:54:23.718849867 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Prediction file: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Shift-TA_TB-to-TC_10pct/50_shot_per_class_stable_lowlr_v1/dumps/ttd/test/coco_predictions_segm.json
Prediction exists: True


,experiment_key,experiment_name,shot_per_class,run_tag,train_status,eval_status,checkpoint,prediction_json,error
0,single_tb_5shot_stable_lowlr_v1,Single-TB,5,stable_lowlr_v1,skipped,skipped,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
1,single_tb_10shot_stable_lowlr_v1,Single-TB,10,stable_lowlr_v1,trained,evaluated,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
2,single_tb_25shot_stable_lowlr_v1,Single-TB,25,stable_lowlr_v1,trained,evaluated,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
3,single_tb_50shot_stable_lowlr_v1,Single-TB,50,stable_lowlr_v1,trained,evaluated,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
4,shift_ta_tc_to_tb_10pct_5shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,5,stable_lowlr_v1,trained,evaluated,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
5,shift_ta_tc_to_tb_10pct_10shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,10,stable_lowlr_v1,trained,evaluated,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
6,shift_ta_tc_to_tb_10pct_25shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,25,stable_lowlr_v1,trained,evaluated,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
7,shift_ta_tc_to_tb_10pct_50shot_stable_lowlr_v1,Shift-TA_TC-to-TB_10pct,50,stable_lowlr_v1,trained,evaluated,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
8,shift_ta_tb_to_tc_10pct_5shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,5,stable_lowlr_v1,skipped,skipped,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None
9,shift_ta_tb_to_tc_10pct_10shot_stable_lowlr_v1,Shift-TA_TB-to-TC_10pct,10,stable_lowlr_v1,trained,evaluated,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,/users/7/yu001011/csci5527/CSCI5527-final/SAM3...,None


## IoU Analysis: How to Read It

COCO AP already uses IoU internally, but it does not show direct overlap statistics. This cell adds a more literal mask-overlap view for the test set.

- **mean_best_iou_gt**: for each ground-truth crack mask, find the best overlapping predicted mask; higher means masks overlap the objects better.
- **gt_recall@0.50 / gt_recall@0.75**: fraction of ground-truth objects that have at least one prediction with mask IoU above the threshold. This answers, “how many real cracks did we find?”
- **pred_precision@0.50 / pred_precision@0.75**: fraction of predictions that match a ground-truth object above the threshold. This answers, “how many predicted cracks were credible?”
- **f1@threshold**: harmonic mean of recall and precision at that IoU threshold.
- **matched_mean_iou@threshold**: mean IoU of one-to-one greedy matches at that threshold; this ignores unmatched false positives/false negatives, so read it together with recall and precision.

A common interpretation is: IoU@0.50 is a loose localization check, IoU@0.75 is a stricter mask-quality check. If recall@0.50 is much higher than recall@0.75, the model is roughly finding objects but mask boundaries/localization are weak.

The score-threshold table is important because the prediction dump keeps many low-confidence masks. Very low precision at `score >= 0.00` can simply mean there are many low-confidence extra masks; check whether precision improves as the score threshold increases.


In [4]:
# IoU Analysis: direct mask-overlap metrics on the test predictions
import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils

if "EXPERIMENT" not in globals():
    raise RuntimeError("Run the Experiment Selection cell first.")

PRED_JSON = RUN_DIR / "dumps" / "ttd" / "test" / "coco_predictions_segm.json"
GT_JSON = RUN_ROOT / "test" / "_annotations.coco.json"

print("Selected experiment:", EXPERIMENT_KEY)
print("Prediction file:", PRED_JSON)
print("Ground truth file:", GT_JSON)

if not PRED_JSON.exists():
    raise FileNotFoundError(f"Prediction file not found. Run the Test Eval cell first: {PRED_JSON}")
if PRED_JSON.stat().st_size <= 2:
    raise ValueError(f"Prediction file is empty. Re-run Test Eval after the config fixes: {PRED_JSON}")

coco_gt = COCO(str(GT_JSON))
coco_dt = coco_gt.loadRes(str(PRED_JSON))

cat_ids = coco_gt.getCatIds()
img_ids = coco_gt.getImgIds()
cat_name = {cat["id"]: cat["name"] for cat in coco_gt.loadCats(cat_ids)}

def ann_to_rle(coco, ann):
    """Return a COCO RLE dict for one annotation segmentation."""
    h, w = coco.imgs[ann["image_id"]]["height"], coco.imgs[ann["image_id"]]["width"]
    seg = ann["segmentation"]
    if isinstance(seg, list):
        rles = mask_utils.frPyObjects(seg, h, w)
        return mask_utils.merge(rles)
    if isinstance(seg.get("counts"), list):
        return mask_utils.frPyObjects(seg, h, w)
    return seg

def greedy_match_ious(iou_matrix, threshold):
    """Greedy one-to-one matching by descending IoU."""
    if iou_matrix.size == 0:
        return []
    pairs = []
    for pred_idx, gt_idx in zip(*np.where(iou_matrix >= threshold)):
        pairs.append((float(iou_matrix[pred_idx, gt_idx]), int(pred_idx), int(gt_idx)))
    pairs.sort(reverse=True)
    used_pred, used_gt, matched = set(), set(), []
    for iou, pred_idx, gt_idx in pairs:
        if pred_idx in used_pred or gt_idx in used_gt:
            continue
        used_pred.add(pred_idx)
        used_gt.add(gt_idx)
        matched.append(iou)
    return matched

def summarize_iou(cat_id=None, score_threshold=0.0, thresholds=(0.50, 0.75)):
    best_iou_per_gt = []
    best_iou_per_pred = []
    matched_by_threshold = {thr: [] for thr in thresholds}
    total_gt = 0
    total_pred = 0
    images_with_gt = 0
    images_with_pred = 0

    cats = [cat_id] if cat_id is not None else cat_ids
    for img_id in img_ids:
        for cur_cat_id in cats:
            gt_anns = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=[img_id], catIds=[cur_cat_id], iscrowd=None))
            dt_anns = coco_dt.loadAnns(coco_dt.getAnnIds(imgIds=[img_id], catIds=[cur_cat_id]))
            dt_anns = [ann for ann in dt_anns if ann.get("score", 0.0) >= score_threshold]
            dt_anns = sorted(dt_anns, key=lambda ann: ann.get("score", 0.0), reverse=True)

            if gt_anns:
                images_with_gt += 1
            if dt_anns:
                images_with_pred += 1
            total_gt += len(gt_anns)
            total_pred += len(dt_anns)

            if not gt_anns and not dt_anns:
                continue
            if not gt_anns:
                best_iou_per_pred.extend([0.0] * len(dt_anns))
                continue
            if not dt_anns:
                best_iou_per_gt.extend([0.0] * len(gt_anns))
                continue

            gt_rles = [ann_to_rle(coco_gt, ann) for ann in gt_anns]
            dt_rles = [ann["segmentation"] for ann in dt_anns]
            iscrowd = [int(ann.get("iscrowd", 0)) for ann in gt_anns]
            ious = mask_utils.iou(dt_rles, gt_rles, iscrowd)

            best_iou_per_gt.extend(ious.max(axis=0).tolist())
            best_iou_per_pred.extend(ious.max(axis=1).tolist())
            for thr in thresholds:
                matched_by_threshold[thr].extend(greedy_match_ious(ious, thr))

    best_iou_per_gt = np.asarray(best_iou_per_gt, dtype=float)
    best_iou_per_pred = np.asarray(best_iou_per_pred, dtype=float)

    row = {
        "category": "all" if cat_id is None else cat_name.get(cat_id, str(cat_id)),
        "score_threshold": score_threshold,
        "images": len(img_ids),
        "images_with_gt": images_with_gt,
        "images_with_pred": images_with_pred,
        "gt_instances": total_gt,
        "pred_instances": total_pred,
        "mean_best_iou_gt": float(best_iou_per_gt.mean()) if total_gt else np.nan,
        "median_best_iou_gt": float(np.median(best_iou_per_gt)) if total_gt else np.nan,
        "mean_best_iou_pred": float(best_iou_per_pred.mean()) if total_pred else np.nan,
    }

    for thr in thresholds:
        matched = matched_by_threshold[thr]
        recall = len(matched) / total_gt if total_gt else np.nan
        precision = len(matched) / total_pred if total_pred else np.nan
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall > 0 else 0.0
        row[f"gt_recall@{thr:.2f}"] = recall
        row[f"pred_precision@{thr:.2f}"] = precision
        row[f"f1@{thr:.2f}"] = f1
        row[f"matched_mean_iou@{thr:.2f}"] = float(np.mean(matched)) if matched else 0.0

    return row, best_iou_per_gt, best_iou_per_pred

def rounded_df(rows):
    df = pd.DataFrame(rows)
    for col in df.columns:
        if df[col].dtype.kind in "fc":
            df[col] = df[col].round(4)
    return df

print(f"Prediction file: {PRED_JSON}")
print(f"Ground truth file: {GT_JSON}")

rows = []
overall_row, best_gt, best_pred = summarize_iou(cat_id=None, score_threshold=0.0)
rows.append(overall_row)
for cat_id in cat_ids:
    row, _, _ = summarize_iou(cat_id=cat_id, score_threshold=0.0)
    rows.append(row)

print("Overall/category IoU summary using all predictions:")
display(rounded_df(rows))

threshold_rows = []
for score_thr in [0.0, 0.01, 0.05, 0.10, 0.25, 0.50]:
    row, _, _ = summarize_iou(cat_id=None, score_threshold=score_thr)
    threshold_rows.append(row)

print("IoU summary at different prediction score thresholds:")
display(rounded_df(threshold_rows))

hist_bins = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
hist_counts, bin_edges = np.histogram(best_gt, bins=hist_bins)
hist_df = pd.DataFrame({
    "best_gt_iou_bin": [f"[{bin_edges[i]:.2f}, {bin_edges[i+1]:.2f})" for i in range(len(hist_counts))],
    "gt_count": hist_counts,
})
hist_df.loc[len(hist_df) - 1, "best_gt_iou_bin"] = f"[{bin_edges[-2]:.2f}, {bin_edges[-1]:.2f}]"
print("Best-IoU distribution over ground-truth instances:")
display(hist_df)


Selected experiment: shift_ta_tb_to_tc_10pct_50shot_stable_lowlr_v1
Prediction file: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Shift-TA_TB-to-TC_10pct/50_shot_per_class_stable_lowlr_v1/dumps/ttd/test/coco_predictions_segm.json
Ground truth file: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/fewshot_data/Shift-TA_TB-to-TC_10pct/50_shot_per_class/test/_annotations.coco.json
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Prediction file: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/outputs/fewshot_runs/Shift-TA_TB-to-TC_10pct/50_shot_per_class_stable_lowlr_v1/dumps/ttd/test/coco_predictions_segm.json
Ground truth file: /users/7/yu001011/csci5527/CSCI5527-final/SAM3/fewshot_data/Shift-TA_TB-to-TC_10pct/50_shot_per_class/test/_annotations.coco.json
Overall/category IoU summary using all predictions:


,category,score_threshold,images,images_with_gt,images_with_pred,gt_instances,pred_instances,mean_best_iou_gt,median_best_iou_gt,mean_best_iou_pred,gt_recall@0.50,pred_precision@0.50,f1@0.50,matched_mean_iou@0.50,gt_recall@0.75,pred_precision@0.75,f1@0.75,matched_mean_iou@0.75
0,all,0.0,38,19,38,19,3800,0.4589,0.4805,0.0406,0.4211,0.0021,0.0042,0.6532,0.1579,0.0008,0.0016,0.7879
1,crack,0.0,38,19,38,19,3800,0.4589,0.4805,0.0406,0.4211,0.0021,0.0042,0.6532,0.1579,0.0008,0.0016,0.7879


IoU summary at different prediction score thresholds:


,category,score_threshold,images,images_with_gt,images_with_pred,gt_instances,pred_instances,mean_best_iou_gt,median_best_iou_gt,mean_best_iou_pred,gt_recall@0.50,pred_precision@0.50,f1@0.50,matched_mean_iou@0.50,gt_recall@0.75,pred_precision@0.75,f1@0.75,matched_mean_iou@0.75
0,all,0.00,38,19,38,19,3800,0.4589,0.4805,0.0406,0.4211,0.0021,0.0042,0.6532,0.1579,0.0008,0.0016,0.7879
1,all,0.01,38,19,10,19,682,0.2305,0.0000,0.1202,0.1579,0.0044,0.0086,0.7017,0.1053,0.0029,0.0057,0.8000
2,all,0.05,38,19,8,19,310,0.1915,0.0000,0.1480,0.1579,0.0097,0.0182,0.6969,0.1053,0.0065,0.0122,0.7927
3,all,0.10,38,19,7,19,86,0.1498,0.0000,0.1942,0.1053,0.0233,0.0381,0.6497,0.0526,0.0116,0.0190,0.7941
4,all,0.25,38,19,6,19,11,0.1047,0.0000,0.2565,0.0526,0.0909,0.0667,0.5052,0.0000,0.0000,0.0000,0.0000
5,all,0.50,38,19,3,19,3,0.0675,0.0000,0.4274,0.0526,0.3333,0.0909,0.5052,0.0000,0.0000,0.0000,0.0000


Best-IoU distribution over ground-truth instances:


,best_gt_iou_bin,gt_count
0,"[0.00, 0.10)",1
1,"[0.10, 0.25)",3
2,"[0.25, 0.50)",7
3,"[0.50, 0.75)",5
4,"[0.75, 0.90)",3
5,"[0.90, 1.00]",0
